# Exploratory Data Analysis

This notebook explores the European electricity datasets ingested from the
Ember Energy API and stored in the Bronze layer on Amazon S3.

The objective is to understand the statistical and temporal characteristics
of the data before defining the final Silver-layer transformations.

The analysis focuses on:

- distributions and descriptive statistics
- electricity generation and demand trends
- emissions and carbon intensity
- differences between countries and energy sources
- seasonal patterns
- potential outliers and unusual observations
- relationships between energy generation, demand and emissions

The installed-capacity dataset is analysed separately because its country,
time and technology coverage differs from the four core datasets.

In [32]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Dev\Projects\EuropeanEnergyDataAnalytics


### Visualization Library

Plotly is used for exploratory visualisation in this project.

Its interactive charts are particularly useful for exploring monthly
time-series data across multiple countries and energy sources.

In [33]:
import json
import boto3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from config.settings import load_config

config = load_config()

session = boto3.Session(
    profile_name=config["aws"]["profile_name"],
    region_name=config["project"]["region"],
)

s3 = session.client("s3")

bucket = config["project"]["bucket"]
bronze_prefix = config["storage"]["bronze_prefix"]

## 1. Load Bronze Data

The five datasets are loaded directly from the Bronze layer in Amazon S3.

The Bronze layer contains the source-faithful API responses. No permanent
transformations are applied at this stage. Date conversion performed in this
notebook is used only for exploratory analysis.

In [34]:
def load_bronze_dataset(dataset_name: str) -> pd.DataFrame:
    prefix = f"{bronze_prefix}/{dataset_name}/"

    response = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=prefix,
    )

    objects = response.get("Contents", [])

    if not objects:
        raise FileNotFoundError(
            f"No Bronze file found for {dataset_name}"
        )

    latest_object = max(
        objects,
        key=lambda obj: obj["LastModified"],
    )

    key = latest_object["Key"]

    response = s3.get_object(
        Bucket=bucket,
        Key=key,
    )

    raw = json.loads(
        response["Body"].read().decode("utf-8")
    )

    return pd.DataFrame(raw["data"])

In [35]:
datasets = {
    name: load_bronze_dataset(name)
    for name in config["datasets"]
}

generation = datasets["generation"].copy()
demand = datasets["demand"].copy()
emissions = datasets["emissions"].copy()
carbon_intensity = datasets["carbon_intensity"].copy()
capacity = datasets["capacity"].copy()

In [36]:
for df in datasets.values():
    df["date"] = pd.to_datetime(df["date"])

In [37]:
generation = datasets["generation"]
demand = datasets["demand"]
emissions = datasets["emissions"]
carbon_intensity = datasets["carbon_intensity"]
capacity = datasets["capacity"]

## 2. Descriptive Statistics

Before visualising the data or applying outlier-detection methods, the
numerical variables are examined using descriptive statistics.

For each metric, the analysis considers:

- number of observations
- mean and standard deviation
- minimum and maximum
- median
- first and third quartiles

This provides an initial understanding of the scale and variability of each
metric and helps identify values that may require closer investigation.

Extreme values are not automatically treated as data errors. In electricity
data, large or unusual observations may result from seasonality, differences
between countries, changes in the energy mix, or genuine market and system
events.

In [38]:
numeric_columns = {
    "generation": [
        "generation_twh",
        "share_of_generation_pct",
    ],
    "demand": [
        "demand_twh",
    ],
    "emissions": [
        "emissions_mtco2",
        "share_of_emissions_pct",
    ],
    "carbon_intensity": [
        "emissions_intensity_gco2_per_kwh",
    ],
    "capacity": [
        "capacity_gw",
        "capacity_w_per_capita",
    ],
}

for name, columns in numeric_columns.items():
    print(f"\n{name.upper()}")
    display(
        datasets[name][columns]
        .describe()
        .T
    )


GENERATION


,count,mean,std,min,25%,50%,75%,max
generation_twh,43900.0,3.869321,7.094330,-10.68,0.25,1.27,4.2700,59.14
share_of_generation_pct,43900.0,32.107039,36.049708,-41.84,3.51,16.54,51.1625,198.18



DEMAND


,count,mean,std,min,25%,50%,75%,max
demand_twh,2820.0,12.423209,11.915956,1.94,4.33,6.835,14.55,57.04



EMISSIONS


,count,mean,std,min,25%,50%,75%,max
emissions_mtco2,43900.0,0.901720,2.665798,0.0,0.01,0.08,0.470,34.12
share_of_emissions_pct,43900.0,28.055003,38.291044,0.0,0.58,4.99,55.255,100.26



CARBON_INTENSITY


,count,mean,std,min,25%,50%,75%,max
emissions_intensity_gco2_per_kwh,2820.0,282.378908,201.995141,17.94,119.195,262.915,398.61,965.04



CAPACITY


,count,mean,std,min,25%,50%,75%,max
capacity_gw,5268.0,12.915163,17.235754,0.0,2.2500,5.960,15.72,129.10
capacity_w_per_capita,5268.0,380.082747,329.265054,0.0,157.0025,271.735,542.64,1693.39


### Initial Statistical Observations

The descriptive statistics reveal substantial differences in scale and
variability across the energy datasets.

#### Electricity Generation

Electricity generation shows considerable variation across countries,
months and generation technologies. The median monthly generation is
1.27 TWh, while the maximum reaches 59.14 TWh.

Negative values occur in both `generation_twh` and
`share_of_generation_pct`. In addition, the generation share reaches
198.18%, substantially above 100%.

These observations require further investigation before being classified
as outliers or data-quality problems. The generation dataset contains
different types of series, including aggregate series and net imports,
which may explain values outside conventional generation-share ranges.

#### Electricity Demand

Monthly electricity demand ranges from 1.94 TWh to 57.04 TWh. The mean
(12.42 TWh) is considerably higher than the median (6.84 TWh), indicating
a right-skewed distribution.

This is plausible in a cross-country dataset because electricity systems
differ substantially in size.

#### Power-Sector Emissions

Emissions are strongly right-skewed. The median observation is only
0.08 MtCO2 compared with a maximum of 34.12 MtCO2.

The maximum share of emissions is 100.26%, slightly above 100%. This value
should be investigated rather than automatically corrected or removed.

#### Carbon Intensity

Carbon intensity ranges from 17.94 to 965.04 gCO2/kWh, with a median of
262.92 gCO2/kWh.

The large standard deviation indicates substantial variation across
countries and time. This may reflect differences in national electricity
mixes as well as temporal changes in fossil and low-carbon generation.

#### Installed Capacity

Installed capacity also exhibits a right-skewed distribution. Capacity
ranges from 0 to 129.10 GW, while capacity per capita reaches
1,693.39 W per capita.

Because the capacity dataset covers only selected wind and solar
technologies and fewer countries than the core datasets, it will be
analysed separately.

### Next Step

The extreme values identified by the descriptive statistics are not
automatically treated as errors.

The next stage examines the distributions and identifies which countries,
dates and energy series are responsible for unusual values. This allows
statistical outliers to be distinguished from legitimate characteristics
of the electricity system.

## 3. Investigation of Unusual Generation Values

The descriptive statistics identified negative generation values and
generation shares above 100%.

Before applying statistical outlier-detection methods, the observations
responsible for these values are examined directly. This is important
because unusual values may have a valid interpretation depending on the
energy series.

In [39]:
unusual_generation = generation[
    (generation["generation_twh"] < 0)
    | (generation["share_of_generation_pct"] < 0)
    | (generation["share_of_generation_pct"] > 100)
].copy()

print(f"Unusual observations: {len(unusual_generation):,}")

display(
    unusual_generation[
        [
            "entity",
            "entity_code",
            "date",
            "series",
            "is_aggregate_series",
            "generation_twh",
            "share_of_generation_pct",
        ]
    ]
    .sort_values(
        "share_of_generation_pct",
        ascending=False,
    )
    .head(30)
)

Unusual observations: 3,210


,entity,entity_code,date,series,is_aggregate_series,generation_twh,share_of_generation_pct
21076,Denmark,DNK,2020-08-01,Demand,True,2.79,198.18
13115,Hungary,HUN,2018-06-01,Demand,True,3.44,183.91
2899,Denmark,DNK,2015-08-01,Demand,True,2.79,181.32
17355,Denmark,DNK,2019-08-01,Demand,True,2.71,179.87
12808,Hungary,HUN,2018-05-01,Demand,True,3.41,177.57
16735,Denmark,DNK,2019-06-01,Demand,True,2.57,175.07
5759,Denmark,DNK,2016-06-01,Demand,True,2.70,174.82
13036,Denmark,DNK,2018-06-01,Demand,True,2.68,172.38
12194,Hungary,HUN,2018-03-01,Demand,True,3.86,171.55
20455,Denmark,DNK,2020-06-01,Demand,True,2.66,171.53


### Investigation of Unusual Generation Values

A total of 3,210 observations satisfy at least one of the following
conditions:

- negative electricity generation
- negative generation share
- generation share above 100%

Inspection of the largest percentage values shows that many observations
above 100% belong to the aggregate `Demand` series. For example, several
observations for Denmark and Hungary report demand-related shares well
above 100%.

This demonstrates that `share_of_generation_pct` cannot be interpreted
uniformly as a conventional technology share for every series contained
in the generation dataset.

The generation dataset combines physical generation technologies with
aggregate and system-level series. Consequently, values outside the
usual 0–100% range are not classified as data-quality errors solely on
the basis of their magnitude.

The unusual observations are therefore analysed by series before any
outlier treatment is considered.

In [40]:
unusual_breakdown = (
    generation
    .assign(
        negative_generation=
            generation["generation_twh"] < 0,

        negative_share=
            generation["share_of_generation_pct"] < 0,

        share_above_100=
            generation["share_of_generation_pct"] > 100,
    )
    .groupby("series")
    .agg(
        observations=("series", "size"),
        negative_generation=("negative_generation", "sum"),
        negative_share=("negative_share", "sum"),
        share_above_100=("share_above_100", "sum"),
        min_generation_twh=("generation_twh", "min"),
        max_generation_twh=("generation_twh", "max"),
        min_share_pct=("share_of_generation_pct", "min"),
        max_share_pct=("share_of_generation_pct", "max"),
    )
)

display(
    unusual_breakdown[
        (unusual_breakdown["negative_generation"] > 0)
        | (unusual_breakdown["negative_share"] > 0)
        | (unusual_breakdown["share_above_100"] > 0)
    ]
)

,observations,negative_generation,negative_share,share_above_100,min_generation_twh,max_generation_twh,min_share_pct,max_share_pct
series,,,,,,,,
Demand,2820,0,0,1620,1.94,57.04,58.16,198.18
Net imports,2820,1194,1199,0,-10.68,6.11,-41.84,98.20
Total generation,2820,0,0,391,1.41,59.14,99.97,100.02


### Interpretation of Unusual Generation Values

The investigation shows that values outside conventional generation-share
ranges are concentrated in three specific series.

#### Demand

The `Demand` series contains 1,620 observations with
`share_of_generation_pct` above 100%, reaching a maximum of 198.18%.

These values should not be interpreted as conventional technology shares.
`Demand` is an aggregate system-level series rather than an individual
generation technology. Therefore, values above 100% are retained and
treated according to their series-specific meaning.

#### Net Imports

`Net imports` contains 1,194 negative generation values and 1,199 negative
share values. Monthly values range from -10.68 TWh to 6.11 TWh.

The concentration of negative observations within this specific series
indicates that negative values are part of the variable's behaviour rather
than sufficient evidence of a data-quality problem. They will therefore
not be removed during Silver transformation.

#### Total Generation

The `Total generation` share ranges only from 99.97% to 100.02%.
The small deviations around 100% are consistent with numerical or rounding
differences and are not considered meaningful outliers.

#### Data Quality Implication

A universal rule requiring `share_of_generation_pct` to lie between
0% and 100% would incorrectly reject valid observations.

Data-quality validation must therefore account for the semantics of each
series rather than applying the same numerical bounds to the entire
generation dataset.

## 4. Distribution of Electricity Generation by Energy Source

The generation dataset contains both individual generation technologies
and aggregate/system-level series.

To make distributions comparable, the following analysis first focuses
on non-aggregate generation series. Aggregate series such as `Clean`,
`Fossil`, `Demand`, and `Total generation` are excluded from this
particular comparison.

Box plots are used to examine the distribution of monthly generation
across energy sources and to identify observations that are statistically
unusual relative to each technology.

Points outside the box-plot whiskers are treated as observations requiring
investigation, not automatically as erroneous data.

In [41]:
non_aggregate_generation = generation[
    ~generation["is_aggregate_series"]
].copy()

print(
    sorted(
        non_aggregate_generation["series"].unique()
    )
)

['Bioenergy', 'Coal', 'Gas', 'Hydro', 'Net imports', 'Nuclear', 'Other fossil', 'Other renewables', 'Solar', 'Wind']


## 4.1 Distribution of Monthly Generation by Energy Source

The analysis now focuses on the ten non-aggregate generation series:

- Bioenergy
- Coal
- Gas
- Hydro
- Net imports
- Nuclear
- Other fossil
- Other renewables
- Solar
- Wind

Aggregate series are excluded because combining aggregate measures with
their underlying components would make the distributions difficult to
interpret.

A box plot is used to compare the distribution of monthly electricity
generation across energy sources and countries.

The box plot highlights:

- the median generation level
- the interquartile range (IQR)
- differences in variability between technologies
- observations located beyond the conventional box-plot whiskers

Observations beyond the whiskers are considered potential statistical
outliers only. They are not automatically classified as data errors.
Cross-country differences, seasonality and changes over time may produce
legitimate extreme observations.

In [42]:
fig = px.box(
    non_aggregate_generation,
    x="series",
    y="generation_twh",
    points="outliers",
    labels={
        "series": "Energy Source",
        "generation_twh": "Monthly Generation (TWh)",
    },
    title="Distribution of Monthly Electricity Generation by Energy Source",
)

fig.update_layout(
    xaxis_title="Energy Source",
    yaxis_title="Monthly Generation (TWh)",
)

fig.show()

KeyboardInterrupt: 

### IQR-Based Outlier Screening

The visual inspection is complemented with an interquartile-range
calculation for each energy source.

For each series, potential outliers are defined as observations below

**Q1 − 1.5 × IQR**

or above

**Q3 + 1.5 × IQR**,

where:

**IQR = Q3 − Q1**

The calculation is performed separately for each energy source because
generation technologies have substantially different distributions.

This screening is exploratory. Flagged observations are retained until
their country, date and temporal context have been investigated.

In [ ]:
outlier_summary = []

for series, group in non_aggregate_generation.groupby("series"):
    q1 = group["generation_twh"].quantile(0.25)
    q3 = group["generation_twh"].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = group[
        (group["generation_twh"] < lower_bound)
        | (group["generation_twh"] > upper_bound)
    ]

    outlier_summary.append({
        "series": series,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": len(outliers),
        "outlier_pct": (
            len(outliers) / len(group) * 100
        ),
    })

outlier_summary = pd.DataFrame(outlier_summary)

display(
    outlier_summary
    .sort_values("outlier_pct", ascending=False)
    .round(2)
)

,series,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
1,Coal,0.13,1.55,1.42,-2.00,3.68,326,13.43
0,Bioenergy,0.12,0.66,0.54,-0.69,1.47,302,12.15
9,Wind,0.44,2.02,1.58,-1.94,4.39,311,11.03
8,Solar,0.09,1.03,0.94,-1.32,2.44,269,10.49
7,Other renewables,0.00,0.16,0.16,-0.24,0.40,150,10.07
2,Gas,0.41,3.45,3.04,-4.15,8.00,259,9.87
4,Net imports,-0.68,0.90,1.58,-3.05,3.27,270,9.57
5,Nuclear,1.28,4.57,3.29,-3.66,9.51,140,8.47
6,Other fossil,0.07,0.34,0.27,-0.34,0.75,183,7.20
3,Hydro,0.08,2.77,2.69,-3.96,6.80,158,5.60


### Interpretation of Generation Distributions and IQR Screening

The box plots reveal strongly right-skewed distributions for several
generation technologies. Coal, nuclear, wind, solar, gas and hydro contain
numerous observations above the conventional box-plot whiskers.

The initial IQR screening identifies relatively large proportions of
potential outliers. For example:

- Coal: 13.43%
- Bioenergy: 12.15%
- Wind: 11.03%
- Solar: 10.49%
- Gas: 9.87%

These proportions are too large to interpret the flagged observations
directly as anomalous or erroneous measurements.

A major reason is the cross-country structure of the dataset. Countries
with large electricity systems naturally generate substantially more
electricity than smaller countries. Consequently, applying an IQR rule
across all countries within an energy source can classify legitimate
country-size differences as statistical outliers.

The initial IQR analysis is therefore useful as a diagnostic step, but
not appropriate as a final outlier rule for this dataset.

A more meaningful analysis should compare observations within the same
country and energy source over time.

## 4.2 Country- and Technology-Specific Outlier Analysis

To control for structural differences in electricity-system size, the
IQR analysis is repeated within each combination of country and energy
source.

An observation is therefore compared only with historical observations
from the same country and the same generation technology.

This provides a more meaningful statistical definition of unusual
monthly generation while preserving legitimate differences between
countries.

The IQR method remains a screening technique rather than an automatic
data-cleaning rule. Flagged observations will subsequently be examined
in their temporal context.

In [ ]:
country_series_outliers = []

for (country, series), group in (
    non_aggregate_generation
    .groupby(["entity_code", "series"])
):
    q1 = group["generation_twh"].quantile(0.25)
    q3 = group["generation_twh"].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    flagged = group[
        (group["generation_twh"] < lower_bound)
        | (group["generation_twh"] > upper_bound)
    ]

    for _, row in flagged.iterrows():
        country_series_outliers.append({
            "entity": row["entity"],
            "entity_code": country,
            "date": row["date"],
            "series": series,
            "generation_twh": row["generation_twh"],
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
        })

country_series_outliers = pd.DataFrame(
    country_series_outliers
)

print(
    f"Potential outliers: "
    f"{len(country_series_outliers):,}"
)

print(
    f"Percentage of non-aggregate observations: "
    f"{len(country_series_outliers) / len(non_aggregate_generation) * 100:.2f}%"
)

Potential outliers: 680
Percentage of non-aggregate observations: 2.80%


In [ ]:
country_series_outliers["distance_from_bound"] = (
    country_series_outliers.apply(
        lambda row:
            row["generation_twh"] - row["upper_bound"]
            if row["generation_twh"] > row["upper_bound"]
            else row["lower_bound"] - row["generation_twh"],
        axis=1,
    )
)

display(
    country_series_outliers
    .sort_values(
        "distance_from_bound",
        ascending=False,
    )
    .head(20)
)

,entity,entity_code,date,series,generation_twh,lower_bound,upper_bound,distance_from_bound
273,United Kingdom,GBR,2015-01-01,Coal,9.89,-1.32500,2.55500,7.33500
275,United Kingdom,GBR,2015-03-01,Coal,9.62,-1.32500,2.55500,7.06500
274,United Kingdom,GBR,2015-02-01,Coal,9.22,-1.32500,2.55500,6.66500
276,United Kingdom,GBR,2015-04-01,Coal,7.12,-1.32500,2.55500,4.56500
282,United Kingdom,GBR,2015-10-01,Coal,6.37,-1.32500,2.55500,3.81500
283,United Kingdom,GBR,2015-11-01,Coal,5.52,-1.32500,2.55500,2.96500
232,Spain,ESP,2026-07-01,Solar,9.14,-2.05875,6.31125,2.82875
254,France,FRA,2015-01-01,Nuclear,43.60,19.36625,40.93625,2.66375
277,United Kingdom,GBR,2015-05-01,Coal,5.07,-1.32500,2.55500,2.51500
286,United Kingdom,GBR,2016-02-01,Coal,4.89,-1.32500,2.55500,2.33500


### Result of Country- and Technology-Specific Outlier Screening

After controlling for both country and generation technology, the IQR
method identifies 680 potential outliers, corresponding to 2.80% of all
non-aggregate generation observations.

This is substantially more selective than the previous technology-level
screening.

The result confirms that many observations previously classified as
outliers were caused by structural differences between national
electricity systems rather than unusual behaviour within a country.

The remaining 2.80% represent months that are statistically unusual
relative to the historical distribution of the same technology within
the same country.

However, these observations are not automatically considered data errors.
Electricity generation is seasonal and can also be affected by weather,
plant availability, policy changes and structural changes in generation
capacity.

The flagged observations therefore require temporal interpretation before
any decision about data treatment is made.

In [ ]:
display(
    country_series_outliers
    .groupby("series")
    .size()
    .sort_values(ascending=False)
    .rename("outlier_count")
    .to_frame()
)

,outlier_count
series,
Solar,210
Other fossil,126
Other renewables,98
Nuclear,57
Coal,49
Bioenergy,41
Hydro,31
Net imports,31
Gas,21


### Distribution of Potential Outliers by Energy Source

The country- and technology-specific IQR screening identifies 680
potential outliers in total.

Solar accounts for the largest number of flagged observations (210),
followed by Other fossil (126) and Other renewables (98). In contrast,
wind generation produces relatively few flagged observations (16).

The high number of solar flags requires particular caution. The IQR method
assumes a relatively stable historical distribution, whereas electricity
generation can exhibit both long-term structural change and strong
seasonality.

For technologies whose generation has changed substantially over the
2010–2026 period, observations from later years may be statistically
different from earlier years without being erroneous.

The next step therefore examines the temporal behaviour of generation
before deciding whether the flagged observations represent anomalies,
seasonality or structural trends.

In [ ]:
country_series_outliers["distance_from_bound"] = (
    country_series_outliers.apply(
        lambda row:
            row["generation_twh"] - row["upper_bound"]
            if row["generation_twh"] > row["upper_bound"]
            else row["lower_bound"] - row["generation_twh"],
        axis=1,
    )
)

display(
    country_series_outliers
    .sort_values("distance_from_bound", ascending=False)
    .head(20)
)

,entity,entity_code,date,series,generation_twh,lower_bound,upper_bound,distance_from_bound
273,United Kingdom,GBR,2015-01-01,Coal,9.89,-1.32500,2.55500,7.33500
275,United Kingdom,GBR,2015-03-01,Coal,9.62,-1.32500,2.55500,7.06500
274,United Kingdom,GBR,2015-02-01,Coal,9.22,-1.32500,2.55500,6.66500
276,United Kingdom,GBR,2015-04-01,Coal,7.12,-1.32500,2.55500,4.56500
282,United Kingdom,GBR,2015-10-01,Coal,6.37,-1.32500,2.55500,3.81500
283,United Kingdom,GBR,2015-11-01,Coal,5.52,-1.32500,2.55500,2.96500
232,Spain,ESP,2026-07-01,Solar,9.14,-2.05875,6.31125,2.82875
254,France,FRA,2015-01-01,Nuclear,43.60,19.36625,40.93625,2.66375
277,United Kingdom,GBR,2015-05-01,Coal,5.07,-1.32500,2.55500,2.51500
286,United Kingdom,GBR,2016-02-01,Coal,4.89,-1.32500,2.55500,2.33500


### Examination of the Most Extreme IQR Flags

Inspection of the observations furthest beyond their country- and
technology-specific IQR bounds reveals an important limitation of the
method.

Several of the strongest flags occur in UK coal generation during
2015–2017. Rather than appearing as isolated observations, these high
values are concentrated in the earlier part of the time series. This
pattern is consistent with a structural change in coal generation over
time rather than random measurement anomalies.

A similar effect appears for solar generation. Several of the largest
solar flags occur in 2026 in Spain, Germany and France. When a technology
changes substantially over the observation period, a single IQR
calculated across the complete historical distribution can classify
later observations as outliers even when they form part of an evolving
trend.

The results demonstrate that statistical outlier detection must be
interpreted together with the temporal structure of the data.

For this reason, the IQR flags will not be used to remove observations
from the Silver layer. They are retained as analytical indicators for
further investigation.

## 4.3 Generation Trends in Germany

To place the statistical outlier results in their temporal context, monthly
electricity generation in Germany is examined for four major energy sources:
solar, wind, coal and gas.

The time-series view helps distinguish isolated anomalies from systematic
patterns such as:

- long-term structural changes
- growth or decline of individual technologies
- recurring seasonal variation
- temporary peaks and troughs

Germany is used as a detailed example rather than producing separate charts
for all 20 countries. Cross-country patterns are examined later using more
compact comparative analyses.

In [ ]:
germany_generation = non_aggregate_generation[
    (non_aggregate_generation["entity_code"] == "DEU")
    & non_aggregate_generation["series"].isin(
        ["Solar", "Wind", "Coal", "Gas"]
    )
].copy()

fig = px.line(
    germany_generation,
    x="date",
    y="generation_twh",
    color="series",
    title="Monthly Electricity Generation in Germany",
    labels={
        "date": "Date",
        "generation_twh": "Monthly Generation (TWh)",
        "series": "Energy Source",
    },
)

fig.show()

### Interpretation of German Generation Trends

The German time series reveals clear structural and seasonal patterns in
electricity generation.

Coal generation shows the strongest long-term decline. Monthly coal
generation was above 20 TWh during parts of 2015 and subsequently decreased
substantially. By 2025–2026, monthly generation is generally below 10 TWh.
This supports the earlier conclusion that unusually high historical coal
observations should not automatically be treated as data errors. They form
part of a long-term structural change in the German electricity mix.

Solar generation exhibits a particularly strong seasonal pattern, with
repeated peaks during the warmer months and troughs during winter. At the
same time, the height of the seasonal peaks increases over the observation
period, indicating a long-term growth trend alongside seasonality. The high
solar observations identified by the IQR analysis are therefore consistent
with an evolving time series rather than isolated anomalies.

Wind generation also shows substantial month-to-month and seasonal
variation. Its peaks become more prominent during the later years, although
the pattern is less regular than for solar.

Gas generation fluctuates considerably but remains within a comparatively
stable range over much of the observation period.

Overall, the chart demonstrates that electricity generation contains both
long-term structural trends and recurring seasonal variation. Consequently,
statistical outlier detection without temporal context would risk
misclassifying legitimate observations as anomalies.

## 4.4 Seasonal Patterns in Electricity Generation

The time-series analysis indicates recurring within-year variation,
particularly for solar and wind generation.

To examine seasonality more directly, monthly generation is grouped by
calendar month. Averaging observations from the same month across multiple
years reduces short-term fluctuations and makes recurring seasonal patterns
easier to identify.

Germany is again used as a detailed example. Solar, wind, coal and gas are
compared to determine whether their monthly generation follows systematic
seasonal patterns.

In [ ]:
germany_seasonality = (
    germany_generation
    .assign(month=germany_generation["date"].dt.month)
    .groupby(["month", "series"], as_index=False)["generation_twh"]
    .mean()
)

fig = px.line(
    germany_seasonality,
    x="month",
    y="generation_twh",
    color="series",
    markers=True,
    title="Average Monthly Generation by Energy Source in Germany",
    labels={
        "month": "Month",
        "generation_twh": "Average Monthly Generation (TWh)",
        "series": "Energy Source",
    },
)

fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(1, 13)),
    ticktext=[
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
    ],
)

fig.show()

### Interpretation of Seasonal Generation Patterns

The monthly averages reveal clear seasonal patterns in German electricity
generation.

Solar generation exhibits the strongest and most regular seasonality.
Average generation increases from approximately 1 TWh in winter to around
8 TWh in early summer before declining again towards the end of the year.

Wind generation shows an approximately opposite seasonal pattern. Average
wind generation is highest during the winter months and substantially lower
during late spring and summer.

The opposing seasonal patterns of solar and wind indicate a degree of
seasonal complementarity between the two renewable technologies: periods
of relatively low solar generation tend to coincide with higher wind
generation, and vice versa.

Coal and gas generation also tend to be higher during the colder months
and lower during spring and summer. Their seasonal variation is less
pronounced than that of solar and wind.

These results confirm that seasonality is an important structural
characteristic of the generation data. Consequently, unusually high or
low monthly observations should be interpreted relative to both the
technology and the time of year rather than being classified as anomalies
solely from their position in the overall distribution.

## 5. Demand, Emissions and Carbon Intensity

After examining the structure of electricity generation, the analysis now
moves to system-level indicators.

Electricity demand describes the amount of electricity required by the
system, while power-sector emissions quantify the resulting CO₂ emissions.
Carbon intensity expresses emissions relative to electricity generation
and therefore provides an additional measure for comparing the environmental
characteristics of electricity systems.

The analysis first examines their development over time before investigating
relationships between these variables and the electricity generation mix.

In [ ]:
germany_system = (
    demand[demand["entity_code"] == "DEU"][
        ["date", "demand_twh"]
    ]
    .merge(
        carbon_intensity[
            carbon_intensity["entity_code"] == "DEU"
        ][["date", "emissions_intensity_gco2_per_kwh"]],
        on="date",
        how="inner",
    )
)

germany_system["year"] = germany_system["date"].dt.year

germany_annual = (
    germany_system
    .groupby("year", as_index=False)
    .agg(
        avg_monthly_demand_twh=("demand_twh", "mean"),
        avg_carbon_intensity=("emissions_intensity_gco2_per_kwh", "mean"),
    )
)

display(germany_annual)

,year,avg_monthly_demand_twh,avg_carbon_intensity
0,2015,46.432500,528.056667
1,2016,46.640000,524.204167
2,2017,46.634167,493.140000
3,2018,45.961667,477.756667
4,2019,45.012500,410.677500
5,2020,43.788333,372.064167
6,2021,45.388333,411.478333
7,2022,43.547500,441.938333
8,2023,40.895833,380.188333
9,2024,41.521667,354.563333


### Interpretation of Demand and Carbon Intensity Trends

The annual averages show different long-term developments in German
electricity demand and carbon intensity.

Average monthly electricity demand changes relatively moderately over the
observation period. It decreases from approximately 46.4 TWh in 2015 to
42.2 TWh in 2026, although the development is not continuous.

Carbon intensity shows a much stronger downward trend. Average intensity
falls from approximately 528 gCO₂/kWh in 2015 to 331 gCO₂/kWh in 2026,
representing a reduction of roughly 37%.

The decline is not monotonic. Carbon intensity temporarily increases in
2021 and 2022 before falling again from 2023 onwards. This indicates that
the long-term reduction contains shorter-term fluctuations that should be
investigated together with changes in the electricity generation mix.

The substantially larger reduction in carbon intensity compared with
electricity demand suggests that changes in demand alone cannot explain
the observed decline in emissions intensity. The composition of electricity
generation is therefore an important variable for the subsequent analysis.

The 2026 values represent January to August only and should therefore be
interpreted as year-to-date averages rather than complete annual values.

In [ ]:
fig = px.line(
    germany_annual,
    x="year",
    y="avg_carbon_intensity",
    markers=True,
    title="Average Carbon Intensity in Germany",
    labels={
        "year": "Year",
        "avg_carbon_intensity": "Carbon Intensity (gCO₂/kWh)",
    },
)

fig.show()

In [ ]:
fig = px.line(
    germany_annual,
    x="year",
    y="avg_monthly_demand_twh",
    markers=True,
    title="Average Monthly Electricity Demand in Germany",
    labels={
        "year": "Year",
        "avg_monthly_demand_twh": "Average Monthly Demand (TWh)",
    },
)

fig.show()

### Visual Interpretation

The visual comparison reinforces the difference between the two long-term
developments.

Electricity demand does not follow a continuous downward trend. After
remaining relatively stable between 2015 and 2017, demand declines,
temporarily recovers in 2021, and reaches its lowest annual average in
2023. A modest recovery is visible afterwards.

Carbon intensity follows a much clearer long-term downward trajectory.
The temporary reversal between 2020 and 2022 interrupts this trend, but
the decline resumes strongly from 2023 onwards.

The different shapes of the two time series provide further evidence that
changes in electricity demand alone are unlikely to explain the substantial
reduction in carbon intensity. This motivates examining the relationship
between the electricity generation mix and carbon intensity.

## 5.1 Generation Mix and Carbon Intensity

The previous analysis showed that German carbon intensity declined much
more strongly than electricity demand.

A plausible explanatory factor is the composition of electricity
generation. Electricity produced from fossil sources is expected to be
associated with higher carbon intensity, whereas a larger share of clean
generation should generally be associated with lower carbon intensity.

The Ember generation dataset already provides aggregate Clean and Fossil
generation series. These source-defined aggregates are used here instead
of constructing new classifications from individual technologies.

Monthly generation shares are merged with monthly carbon intensity to
investigate these relationships.

In [ ]:
germany_mix = generation[
    (generation["entity_code"] == "DEU")
    & generation["series"].isin(["Clean", "Fossil"])
][
    ["date", "series", "share_of_generation_pct"]
].copy()

germany_mix = (
    germany_mix
    .pivot(
        index="date",
        columns="series",
        values="share_of_generation_pct",
    )
    .reset_index()
    .rename(
        columns={
            "Clean": "clean_share_pct",
            "Fossil": "fossil_share_pct",
        }
    )
)

germany_mix_carbon = germany_mix.merge(
    carbon_intensity[
        carbon_intensity["entity_code"] == "DEU"
    ][
        ["date", "emissions_intensity_gco2_per_kwh"]
    ],
    on="date",
    how="inner",
)

display(germany_mix_carbon.head())

,date,clean_share_pct,fossil_share_pct,emissions_intensity_gco2_per_kwh
0,2015-01-01,44.98,55.02,514.60
1,2015-02-01,37.97,62.04,572.74
2,2015-03-01,42.77,57.23,540.54
3,2015-04-01,47.28,52.72,513.50
4,2015-05-01,52.40,47.60,461.76


In [ ]:
correlations = germany_mix_carbon[
    [
        "clean_share_pct",
        "fossil_share_pct",
        "emissions_intensity_gco2_per_kwh",
    ]
].corr()

display(correlations)

,clean_share_pct,fossil_share_pct,emissions_intensity_gco2_per_kwh
clean_share_pct,1.000000,-1.000000,-0.961914
fossil_share_pct,-1.000000,1.000000,0.961928
emissions_intensity_gco2_per_kwh,-0.961914,0.961928,1.000000


### Interpretation of Generation Mix and Carbon Intensity

The correlation analysis reveals a very strong relationship between the
German electricity generation mix and carbon intensity.

The fossil generation share has a strong positive correlation with carbon
intensity (r ≈ 0.962). Months with a larger share of fossil electricity
generation therefore tend to have substantially higher carbon intensity.

Conversely, the clean generation share has a strong negative correlation
with carbon intensity (r ≈ -0.962). Higher shares of clean electricity
generation tend to coincide with lower emissions intensity.

Clean and fossil generation shares are perfectly negatively correlated
(r = -1.0). This indicates that the two aggregate shares are complementary
components of the generation mix and contain essentially the same
information in inverse form.

Consequently, both variables should not be treated as independent predictors
in a statistical model because this would introduce perfect multicollinearity.

The correlation results provide strong evidence of association, but they do
not by themselves establish causality.

In [ ]:
fig = px.scatter(
    germany_mix_carbon,
    x="fossil_share_pct",
    y="emissions_intensity_gco2_per_kwh",
    trendline="ols",
    title="Fossil Generation Share vs Carbon Intensity in Germany",
    labels={
        "fossil_share_pct": "Fossil Share of Generation (%)",
        "emissions_intensity_gco2_per_kwh": "Carbon Intensity (gCO₂/kWh)",
    },
)

fig.show()

### Interpretation of the Fossil Share Relationship

The scatter plot confirms the strong positive relationship identified by
the correlation analysis.

Monthly observations are concentrated relatively closely around the fitted
regression line. Higher fossil shares are consistently associated with
higher carbon intensity in the German electricity system.

The Pearson correlation coefficient of approximately 0.962 indicates a very
strong linear association between the two variables.

However, the observations do not lie exactly on the regression line. This
remaining variation indicates that fossil generation share alone does not
fully describe monthly carbon intensity. Differences in the composition of
fossil generation and other characteristics of the electricity system may
also contribute to the observed variation.

The regression line is used here as an exploratory description of the
relationship and should not be interpreted as evidence that the observed
association is causal.

## 6. Cross-Country Comparison

The detailed analysis of Germany demonstrated how generation structure,
seasonality and carbon intensity can be investigated at country level.

To extend the analysis to the European scope of the project, carbon
intensity is compared across all countries in the dataset.

The year 2025 is used because it is the latest complete calendar year.
Using a complete year avoids comparing full-year observations with the
partial 2026 data available through August.

In [ ]:
country_carbon_2025 = (
    carbon_intensity[
        carbon_intensity["date"].dt.year == 2025
    ]
    .groupby(
        ["entity", "entity_code"],
        as_index=False,
    )
    .agg(
        avg_carbon_intensity=(
            "emissions_intensity_gco2_per_kwh",
            "mean",
        )
    )
    .sort_values("avg_carbon_intensity")
)

display(country_carbon_2025)

,entity,entity_code,avg_carbon_intensity
17,Sweden,SWE,23.882500
12,Norway,NOR,27.071667
5,France,FRA,32.382500
18,Switzerland,CHE,58.110000
4,Finland,FIN,71.228333
0,Austria,AUT,95.809167
14,Portugal,PRT,125.142500
16,Spain,ESP,138.459167
3,Denmark,DNK,140.098333
1,Belgium,BEL,160.295000


In [ ]:
fig = px.bar(
    country_carbon_2025,
    x="avg_carbon_intensity",
    y="entity",
    orientation="h",
    title="Average Power-Sector Carbon Intensity by Country, 2025",
    labels={
        "avg_carbon_intensity": "Average Carbon Intensity (gCO₂/kWh)",
        "entity": "Country",
    },
)

fig.show()

### Interpretation of Cross-Country Carbon Intensity

The 2025 comparison reveals substantial differences in power-sector carbon
intensity across the European countries included in the dataset.

Carbon intensity ranges from only a few tens of gCO₂/kWh among the
lowest-intensity electricity systems to more than 600 gCO₂/kWh at the
upper end of the distribution.

Germany, with an annual average of approximately 348 gCO₂/kWh, lies in the
higher part of the observed country distribution in 2025.

The wide cross-country variation demonstrates that national electricity
systems differ substantially in their emissions intensity. This supports
the importance of retaining the country dimension throughout the analytical
pipeline rather than relying only on European-level aggregates.

The comparison itself does not establish why individual countries have
higher or lower carbon intensity. Explaining these differences requires
combining carbon-intensity data with the corresponding national generation
mix and potentially other system characteristics.

## 7. Installed Capacity

Installed capacity is treated as a supplementary dataset because its
coverage differs from the four core datasets.

The available capacity data covers 12 of the 20 project countries from
2016 onwards and focuses on wind- and solar-related technologies.
Consequently, it should not be interpreted as a complete representation
of national electricity-generation capacity.

The EDA therefore focuses on identifying the long-term development of the
available renewable capacity rather than making complete cross-country
capacity comparisons.

In [43]:
germany_capacity = capacity[
    (capacity["entity_code"] == "DEU")
    & capacity["series"].isin(
        ["Solar", "Onshore wind", "Offshore wind"]
    )
].copy()

fig = px.line(
    germany_capacity,
    x="date",
    y="capacity_gw",
    color="series",
    title="Installed Renewable Capacity in Germany",
    labels={
        "date": "Date",
        "capacity_gw": "Installed Capacity (GW)",
        "series": "Technology",
    },
)

fig.show()

### Interpretation of Installed Renewable Capacity

The German capacity data shows substantial growth in renewable electricity
infrastructure between 2016 and 2026.

Solar capacity exhibits the strongest expansion. Installed capacity rises
from approximately 40 GW in 2016 to almost 130 GW by 2026, with particularly
rapid growth during the later years of the observation period.

Onshore wind capacity also increases over time, although its growth is more
gradual. Offshore wind remains considerably smaller in absolute capacity and
shows a more stepwise development.

The strong expansion of solar capacity is consistent with the generation
EDA, where increasingly high solar-generation peaks were observed in recent
years. This provides additional evidence that these high generation values
represent structural development rather than data anomalies.

However, the capacity dataset has more limited country and technology
coverage than the core datasets. It is therefore retained as a supplementary
analytical dataset rather than used as a complete representation of European
generation capacity.

## 8. EDA Conclusions and Implications for the Silver Layer

The exploratory analysis identified several important structural
characteristics of the Ember electricity data.

### Main analytical findings

- Electricity generation differs substantially across countries and
  technologies, making global outlier thresholds inappropriate.
- Country- and technology-specific IQR screening flagged only 2.80% of
  non-aggregate generation observations.
- Investigation of these observations showed that many statistical
  outliers are associated with long-term structural change or seasonality
  rather than data-quality problems.
- German coal generation shows a pronounced long-term decline, while solar
  generation and installed solar capacity show substantial growth.
- Solar and wind generation exhibit strong but approximately opposing
  seasonal patterns in Germany.
- German carbon intensity declined substantially over the observation
  period despite a much smaller change in electricity demand.
- Fossil generation share is strongly positively associated with carbon
  intensity (r ≈ 0.962), while clean generation share shows the inverse
  relationship.
- Large differences in carbon intensity exist between the European
  electricity systems included in the project.
- Installed-capacity data provides useful supplementary information but has
  more limited country, time and technology coverage than the core datasets.

### Implications for data transformation

The EDA leads to the following decisions for the Silver layer:

1. Statistical outliers will not be automatically removed.
2. Negative net-import values will be preserved because they have valid
   domain meaning.
3. Generation-share values will not be globally restricted to the
   interval 0–100 because aggregate series have different semantics.
4. Aggregate-series indicators will be preserved to prevent accidental
   double counting and to distinguish source-defined aggregates from
   individual technologies.
5. Dates and numerical columns will be converted to explicit analytical
   data types.
6. Business-key uniqueness and required-field completeness will be enforced.
7. The four core datasets will retain the common 20-country analytical
   scope from 2010 onwards.
8. Capacity will remain a separate supplementary dataset with its original
   coverage rather than being artificially completed or imputed.
9. Missing observations will not be artificially filled during Silver
   transformation.
10. Partial-year data, such as 2026, will be retained but must be identified
    correctly when annual aggregations are created.

These decisions establish the transformation and data-quality requirements
for the Bronze-to-Silver pipeline.

In [48]:
from src.transformation.clean_energy_data import clean_energy_data


silver_datasets = {}

for dataset_name, bronze_df in datasets.items():
    silver_df = clean_energy_data(
        bronze_df,
        dataset_name,
    )

    silver_datasets[dataset_name] = silver_df

    print(f"\nDataset: {dataset_name}")
    print(f"Bronze shape: {bronze_df.shape}")
    print(f"Silver shape: {silver_df.shape}")

    print(
        "Date range:",
        silver_df["date"].min(),
        "→",
        silver_df["date"].max(),
    )

    print("\nDtypes:")
    print(silver_df.dtypes)

    print("-" * 60)


Dataset: generation
Bronze shape: (43900, 8)
Silver shape: (43900, 8)
Date range: 2010-01-01 00:00:00 → 2026-08-01 00:00:00

Dtypes:
entity                             string
entity_code                        string
is_aggregate_entity               boolean
date                       datetime64[us]
series                             string
is_aggregate_series               boolean
generation_twh                    float64
share_of_generation_pct           float64
dtype: object
------------------------------------------------------------

Dataset: demand
Bronze shape: (2820, 5)
Silver shape: (2820, 5)
Date range: 2010-01-01 00:00:00 → 2026-08-01 00:00:00

Dtypes:
entity                         string
entity_code                    string
is_aggregate_entity              bool
date                   datetime64[us]
demand_twh                    float64
dtype: object
------------------------------------------------------------

Dataset: emissions
Bronze shape: (43900, 8)
Silver shape: (43

In [45]:
for dataset_name, bronze_df in datasets.items():
    silver_df = silver_datasets[dataset_name]

    removed_rows = len(bronze_df) - len(silver_df)

    print(
        f"{dataset_name}: "
        f"{removed_rows:,} rows removed"
    )

generation: 0 rows removed
demand: 0 rows removed
emissions: 0 rows removed
carbon_intensity: 0 rows removed
capacity: 0 rows removed


In [47]:
from src.quality.data_quality import validate_dataset


EXPECTED_COUNTRIES = {
    "DEU", "FRA", "ESP", "ITA", "GBR",
    "NLD", "BEL", "AUT", "POL", "CZE",
    "DNK", "SWE", "NOR", "FIN", "PRT",
    "IRL", "GRC", "ROU", "HUN", "CHE",
}


silver_quality_results = {}

for dataset_name, silver_df in silver_datasets.items():

    result = validate_dataset(
        silver_df,
        dataset_name,
        EXPECTED_COUNTRIES,
    )

    silver_quality_results[dataset_name] = result

    print(f"\nDataset: {dataset_name}")
    print(f"Passed: {result['passed']}")
    print(f"Duplicate keys: {result['duplicate_keys']}")
    print(f"Unexpected countries: {result['unexpected_countries']}")
    print(f"Null counts: {result['null_counts']}")
    print("-" * 60)


Dataset: generation
Passed: True
Duplicate keys: 0
Unexpected countries: []
Null counts: {'entity_code': 0, 'date': 0, 'series': 0, 'generation_twh': 0, 'share_of_generation_pct': 0}
------------------------------------------------------------

Dataset: demand
Passed: True
Duplicate keys: 0
Unexpected countries: []
Null counts: {'entity_code': 0, 'date': 0, 'demand_twh': 0}
------------------------------------------------------------

Dataset: emissions
Passed: True
Duplicate keys: 0
Unexpected countries: []
Null counts: {'entity_code': 0, 'date': 0, 'series': 0, 'emissions_mtco2': 0, 'share_of_emissions_pct': 0}
------------------------------------------------------------

Dataset: carbon_intensity
Passed: True
Duplicate keys: 0
Unexpected countries: []
Null counts: {'entity_code': 0, 'date': 0, 'emissions_intensity_gco2_per_kwh': 0}
------------------------------------------------------------

Dataset: capacity
Passed: True
Duplicate keys: 0
Unexpected countries: []
Null counts: {'e